# 03 - Validação cruzada e estimativas regionais (Partes 11 e 12)

Escolhe o modelo por **previsão fora da amostra** e estima o share de Lula (e de Flávio, 1 − share) por região.
Dentro da amostra, um binomial com ligação canônica e dummies de região reproduz os totais regionais exatamente (notebook 02, célula 3), então a
estimativa regional só tem conteúdo se vier de um modelo que não viu os dados que prevê.

## Parte 11 - Validação cruzada

**Regra de decisão (fixada antes de ver os resultados).**
1. *Métrica primária*: erro absoluto médio do share de cada UF, em p.p., na validação *leave-one-UF-out* (LOSO), média sobre as 27 UFs. Imita prever um estado
   inédito, e o alvo final são shares regionais (soma de UFs), não municípios. O share da UF é ponderado por votos.
2. *Regra de 1 erro-padrão*: o erro-padrão (EP) é o desvio dos 27 erros por UF do melhor modelo dividido por √27. Todo modelo com erro médio até
   (melhor + EP) é considerado empatado; entre os empatados fica **o de menos parâmetros** (parcimônia).
3. *Métrica secundária*: MAE municipal ponderado por votos na validação 5-fold (semente 0), usada para desempatar quando há igualdade de parâmetros e como verificação.
4. LOSO é a validação honesta porque a autocorrelação espacial é forte (I de Moran do share 2026 = 0,896): em 5-fold aleatório, os vizinhos de um município de teste
   estão no treino e o erro sai otimista.
5. Modelos com efeito fixo de UF (`quasi_uf`, `bb_uf`) só entram no 5-fold: a UF retida não tem efeito próprio no treino. `quasi_logit` dá o mesmo π̂ de `bin_logit`
   (mesmo estimador de β; muda só a variância) e é avaliado uma vez, como `bin_logit`.

In [1]:
import sys, json, warnings, time
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor
from multiprocessing import get_context
import numpy as np, pandas as pd
import plotly.graph_objects as go
warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from src import especificacoes as E, modelos as M
pd.options.display.float_format = "{:,.4f}".format
T0 = time.time()
RECALCULAR = False   # True: refaz a validação; False: usa results/cv/*.parquet quando existir
TAB, FIG, CV = ROOT / "results/tables", ROOT / "results/figures", ROOT / "results/cv"
for p in (TAB, FIG, CV): p.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / "data/processed/municipios.parquet").reset_index(drop=True)
mapa = E.MAPA_2026
E.registrar_backward(json.loads((TAB / "selecao_variaveis.json").read_text())["backward_bic"])
E.definir_dados_cv(df, mapa)
y, n = df[mapa["y"]].to_numpy(float), df[mapa["n"]].to_numpy(float)
uf, reg = df[mapa["uf"]].astype(str).to_numpy(), df[mapa["regiao"]].astype(str).to_numpy()
UFS_ = sorted(set(uf)); REGS = sorted(set(reg))
print(len(df), "municípios modelados;", f"{y.sum()/n.sum():.4%} de votos em Lula;", len(UFS_), "UFs;", len(REGS), "regiões")
NOMES = [k for k in E.CANDIDATOS if k != "quasi_logit"]
NOMES_LOSO = [k for k in NOMES if k not in ("quasi_uf", "bb_uf")]
print(len(NOMES), "candidatos distintos;", len(NOMES_LOSO), "em LOSO")

5570 municípios modelados; 51.7796% de votos em Lula; 27 UFs; 5 regiões
20 candidatos distintos; 18 em LOSO


**Município fora do modelo.** Os totais oficiais (`resultado_regional.csv`, `resultado_uf.csv`) incluem Boa Esperança do Norte (MT), sem voto de 2022 e portanto sem a defasagem exigida (945 votos de Lula e 2.353 de Flávio em 2026).
Os 5.570 municípios acima são os modelados; **todas as comparações com o "real" usam os totais recalculados sobre essas linhas** (verificação na célula 14: os CSVs oficiais usados aqui já coincidem com as linhas modeladas; a célula mede o efeito do município de fora).

In [2]:
folds5 = M.folds_kfold(len(df), k=5, seed=0)
foldsL = M.folds_por_grupo(uf)
def _cv(nome, esquema, folds):
    arq = CV / f"{esquema}_{nome}.parquet"
    if arq.exists() and not RECALCULAR:
        return pd.read_parquet(arq)["pi"].to_numpy()
    return None
def rodar(nomes, esquema, folds):
    pend = [k for k in nomes if _cv(k, esquema, folds) is None]
    saida = {}
    if pend:
        with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
            fut = {(k, i): ex.submit(E.prever_fold, k, tr, te) for k in pend for i, (tr, te) in enumerate(folds)}
            for k in pend:
                pi = np.full(len(df), np.nan)
                for i, (tr, te) in enumerate(folds): pi[te] = fut[(k, i)].result()
                pd.DataFrame({"pi": pi}).to_parquet(CV / f"{esquema}_{k}.parquet"); saida[k] = pi
    for k in nomes:
        if k not in saida: saida[k] = _cv(k, esquema, folds)
    return saida
PI5 = rodar(NOMES, "kfold", folds5)
print("5-fold:", round(time.time() - T0), "s"); PIL = rodar(NOMES_LOSO, "loso", foldsL)
print("LOSO:", round(time.time() - T0), "s")
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    K = dict(zip(NOMES, ex.map(E.parametros_ajuste, NOMES)))
print("NaN nas previsões:", {k: int(np.isnan(v).sum()) for d in (PI5, PIL) for k, v in d.items() if np.isnan(v).any()} or "nenhum")

5-fold: 0 s
LOSO: 0 s


NaN nas previsões: nenhum


**Métricas.** Para cada modelo, sobre as previsões fora da amostra empilhadas: tudo que `metricas` devolve (MAE e RMSE ponderados por votos, MAE simples, deviance binomial, erro nacional)
mais o erro absoluto do share de cada UF e de cada região (p.p.). A dispersão vem dos 5 folds (5-fold) e das 27 UFs (LOSO); EP = desvio / √(número de unidades).

In [3]:
obs_uf = pd.Series(y).groupby(uf).sum() / pd.Series(n).groupby(uf).sum()
obs_reg = pd.Series(y).groupby(reg).sum() / pd.Series(n).groupby(reg).sum()
def erros(pi, esq, folds):
    a_uf = (M.agregar(pi, n, uf).drop("Brasil") - obs_uf).abs() * 100
    a_rg = (M.agregar(pi, n, reg).drop("Brasil") - obs_reg).abs() * 100
    m = M.metricas(y, n, pi)
    if esq == "5-fold": por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    else: por = [M.metricas(y[te], n[te], pi[te])["mae_pond_pp"] for _, te in folds]
    m.update(mae_uf_pp=a_uf.mean(), se_uf_pp=a_uf.std(ddof=1) / np.sqrt(len(a_uf)),
             mae_reg_pp=a_rg.mean(), max_reg_pp=a_rg.max(),
             se_mae_pond_pp=np.std(por, ddof=1) / np.sqrt(len(por)))
    return m, a_uf, a_rg
def tabela(PI, esq, folds):
    L, UFE = [], []
    for k, pi in PI.items():
        m, a_uf, _ = erros(pi, esq, folds)
        L.append({"modelo": k, "familia": E.CANDIDATOS[k].familia, "k": K[k], **m})
        if esq == "LOSO":
            pu = M.agregar(pi, n, uf).drop("Brasil")
            mu = pd.Series({u: M.metricas(y[uf == u], n[uf == u], pi[uf == u])["mae_pond_pp"] for u in UFS_})
            UFE.append(pd.DataFrame({"modelo": k, "uf": UFS_, "obs_pct": obs_uf[UFS_].values * 100, "prev_pct": pu[UFS_].values * 100,
                                     "erro_uf_pp": (pu[UFS_] - obs_uf[UFS_]).values * 100, "mae_pond_municipal_pp": mu[UFS_].values}))
    t = pd.DataFrame(L)
    return t, (pd.concat(UFE) if UFE else None)
T5, _ = tabela(PI5, "5-fold", folds5)
TL, TUF = tabela(PIL, "LOSO", foldsL)
T5.sort_values("mae_pond_pp").to_csv(TAB / "cv_kfold.csv", index=False)
TL.sort_values("mae_uf_pp").to_csv(TAB / "cv_loso.csv", index=False)
TUF.to_csv(TAB / "cv_uf_loso.csv", index=False)
col = ["modelo", "k", "mae_uf_pp", "se_uf_pp", "mae_reg_pp", "mae_pond_pp", "rmse_pond_pp", "deviance", "erro_nacional_pp"]
print("LOSO (ordenado pelo erro médio por UF)"); print(TL.sort_values("mae_uf_pp")[col].to_string(index=False))
print("\n5-fold (ordenado pelo MAE ponderado)"); print(T5.sort_values("mae_pond_pp")[col[:2] + ["mae_pond_pp", "se_mae_pond_pp", "mae_uf_pp", "mae_reg_pp", "rmse_pond_pp", "deviance", "erro_nacional_pp"]].to_string(index=False))

LOSO (ordenado pelo erro médio por UF)
      modelo  k  mae_uf_pp  se_uf_pp  mae_reg_pp  mae_pond_pp  rmse_pond_pp     deviance  erro_nacional_pp
  bin_probit 25     0.9301    0.1304      0.0752       1.2578        1.6686 158,358.0072            0.0948
  wls_elogit 26     0.9388    0.1297      0.0781       1.2546        1.6596 155,354.8002            0.0939
   bin_logit 25     0.9429    0.1295      0.0830       1.2567        1.6614 155,730.0115            0.1000
quasi_spline 29     0.9549    0.1299      0.0915       1.2768        1.6791 158,845.1206            0.0994
  ols_elogit 26     0.9690    0.1380      0.3468       1.3134        1.7062 163,819.7269            0.1984
     betareg 26     0.9754    0.1400      0.3453       1.3059        1.7006 162,798.0921            0.1922
    bb_logit 26     0.9845    0.1393      0.3440       1.3100        1.7010 162,836.5388            0.1990
quasi_reglag 30     0.9963    0.1309      0.0718       1.2812        1.6857 160,315.1001            0.071

**Aplicação da regra.** Considera-se empatado com o melhor todo modelo cujo erro médio por UF fica dentro de 1 EP do melhor; escolhe-se o de menos parâmetros (desempate: MAE ponderado no 5-fold).
O vice é o melhor modelo por erro médio quando ele não é o escolhido; caso contrário, o segundo colocado.

In [4]:
ordem = TL.sort_values("mae_uf_pp").reset_index(drop=True)
melhor, se = ordem.loc[0, "mae_uf_pp"], ordem.loc[0, "se_uf_pp"]
emp = ordem[ordem.mae_uf_pp <= melhor + se].merge(T5[["modelo", "mae_pond_pp"]], on="modelo", suffixes=("", "_5f"))
ESC = emp.sort_values(["k", "mae_pond_pp_5f" if "mae_pond_pp_5f" in emp else "mae_pond_pp"]).iloc[0]["modelo"]
VICE = ordem.loc[0, "modelo"] if ordem.loc[0, "modelo"] != ESC else ordem.loc[1, "modelo"]
print(f"melhor por erro médio: {ordem.loc[0,'modelo']} = {melhor:.3f} p.p. (EP {se:.3f}); limite 1-EP = {melhor+se:.3f}")
print("empatados (dentro de 1 EP):", list(emp.modelo))
print("ESCOLHIDO:", ESC, "| VICE:", VICE)
fig = go.Figure(go.Scatter(x=ordem.mae_uf_pp, y=ordem.modelo, mode="markers", error_x=dict(array=ordem.se_uf_pp), marker=dict(size=8, color="#1b6ca8")))
fig.add_vline(x=melhor + se, line_dash="dash", line_color="#888", annotation_text="melhor + 1 EP")
fig.update_layout(title="LOSO: erro médio do share da UF (p.p.), barras = ±1 EP", xaxis_title="p.p.", yaxis=dict(autorange="reversed"), height=520, template="plotly_white")
fig.write_html(FIG / "cv_loso_mae_uf.html", include_plotlyjs="cdn")
o5 = T5.sort_values("mae_pond_pp")
fig = go.Figure(go.Scatter(x=o5.mae_pond_pp, y=o5.modelo, mode="markers", error_x=dict(array=o5.se_mae_pond_pp), marker=dict(size=8, color="#c2571a")))
fig.update_layout(title="5-fold: MAE municipal ponderado por votos (p.p.), ±1 EP dos folds", xaxis_title="p.p.", yaxis=dict(autorange="reversed"), height=520, template="plotly_white")
fig.write_html(FIG / "cv_kfold_mae.html", include_plotlyjs="cdn")

melhor por erro médio: bin_probit = 0.930 p.p. (EP 0.130); limite 1-EP = 1.061
empatados (dentro de 1 EP): ['bin_probit', 'wls_elogit', 'bin_logit', 'quasi_spline', 'ols_elogit', 'betareg', 'bb_logit', 'quasi_reglag', 'bb_backward', 'bb_spline', 'bin_cauchit', 'glmm_uf']
ESCOLHIDO: bb_backward | VICE: bin_probit


In [5]:
def calib(pi, q=20):
    d = pd.DataFrame({"pi": pi, "y": y, "n": n}); d["b"] = pd.qcut(d.pi, q, duplicates="drop")
    g = d.assign(v=d.pi * d.n).groupby("b", observed=True)[["v", "y", "n"]].sum()
    return g.v / g.n, g.y / g.n
outro = [k for k in ordem.modelo if k not in (ESC, VICE)]
mostrar = [ESC, VICE] + [outro[-1]]   # o pior da lista como contraste
fig = go.Figure(go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line=dict(color="#999", dash="dash"), name="calibração perfeita"))
for k, c in zip(mostrar, ["#1b6ca8", "#c2571a", "#4c9a2a"]):
    p, o = calib(PIL[k]); fig.add_trace(go.Scatter(x=p, y=o, mode="lines+markers", name=k, line=dict(color=c)))
fig.update_layout(title="Calibração LOSO: π̂ médio (ponderado por votos) vs share observado, 20 faixas de π̂", xaxis_title="π̂ previsto", yaxis_title="share observado",
                  template="plotly_white", height=520)
fig.write_html(FIG / "cv_calibracao.html", include_plotlyjs="cdn")
for k in mostrar:
    p, o = calib(PIL[k]); print(k, "| máx |previsto - observado| nas faixas (p.p.):", round(float((p - o).abs().max() * 100), 2))

bb_backward | máx |previsto - observado| nas faixas (p.p.): 0.73
bin_probit | máx |previsto - observado| nas faixas (p.p.): 0.87
l1_logit | máx |previsto - observado| nas faixas (p.p.): 3.32


_(conclusões preenchidas após a execução)_

## Parte 12 - Estimativas regionais

**Procedimento.** Para cada UF, o modelo escolhido é reajustado sem ela (LOSO) e a prevê; a estimativa regional é a soma dos votos previstos das UFs da região dividida pelos votos válidos.
**Intervalos**: para cada UF retida, B = 1.000 sorteios de (i) β ~ N(β̂, Σ̂) do ajuste sem a UF, (ii) um choque de UF δ ~ N(0, σ²) no preditor logit, (iii) votos ~ BetaBin(n, π, θ) com θ do ajuste.
Os sorteios são somados por região e para o Brasil, e tomam-se os quantis 2,5/50/97,5%.

**Escolha de σ.** A alternativa seria σ ≈ 0,05 do GLMM (`glmm_uf`), mas esse σ mede a variação entre UFs *depois* de dar a cada UF seu próprio efeito, com todos os dados. O LOSO erra por mais:
a UF retida não tem efeito próprio e o efeito regional é extrapolado. Usamos o **desvio quadrático médio (RMS) dos resíduos LOSO das UFs no logit**, que é o erro que de fato se comete ao prever um estado inédito.
Custo: σ é estimado nas mesmas 27 UFs em que a cobertura é conferida, então a cobertura por UF é otimista. Mostramos também σ = 0 e σ do GLMM.

In [6]:
from scipy.special import logit
CAND = E.CANDIDATOS[ESC]
def parametros(nome_uf):
    tr = np.flatnonzero(uf != nome_uf); te = np.flatnonzero(uf == nome_uf)
    a = CAND.ajustar(df.iloc[tr], mapa)
    res = a.modelo; X = E.montar_X(df.iloc[te], mapa, a.pad, CAND.termos)
    off = None
    if CAND.termos.lag == "offset":
        off = X[E._COL_OFFSET].to_numpy(float); X = X.drop(columns=E._COL_OFFSET)
    p = np.asarray(res.params, float); C = np.asarray(res.cov_params(), float)
    if CAND.familia == "betabin": theta, p, C = float(np.exp(p[-1])), p[:-1], C[:-1, :-1]
    else: theta = None
    return dict(beta=p, cov=C, X=X.to_numpy(float), off=off, theta=theta, te=te)
t0 = time.time()
with ProcessPoolExecutor(max_workers=14, mp_context=get_context("fork")) as ex:
    PAR = dict(zip(UFS_, ex.map(parametros, UFS_)))
print("ajustes LOSO do modelo escolhido:", ESC, round(time.time() - t0), "s | família:", CAND.familia,
      "| theta médio:", np.round(np.mean([v["theta"] for v in PAR.values() if v["theta"]]), 1) if CAND.familia == "betabin" else "-")
pi_loso = PIL[ESC]
prev_uf = M.agregar(pi_loso, n, uf).drop("Brasil")
res_logit = logit(prev_uf[UFS_].values) - logit(obs_uf[UFS_].values)
SIG_RMS = float(np.sqrt(np.mean(res_logit ** 2)))
SIG_GLMM = float(E.CANDIDATOS["glmm_uf"].ajustar(df, mapa).extra["sigma_uf"])
print(f"σ (RMS dos resíduos LOSO no logit) = {SIG_RMS:.4f} | viés médio = {res_logit.mean():+.4f} | σ do GLMM = {SIG_GLMM:.4f}")

ajustes LOSO do modelo escolhido: bb_backward 11 s | família: betabin | theta médio: 629.9
σ (RMS dos resíduos LOSO no logit) = 0.0536 | viés médio = -0.0187 | σ do GLMM = 0.0532


In [7]:
def simular(sigma, B=1000, seed=7):
    rng = np.random.default_rng(seed)
    return np.column_stack([M.simular_votos_grupo(v["beta"], v["cov"], v["X"], n[v["te"]], np.zeros(len(v["te"])), link=E.LIGACOES[CAND.ligacao],
                            theta=v["theta"], sigma=sigma, B=B, rng=rng, offset=v["off"])[:, 0] for u, v in PAR.items()])
n_uf = pd.Series(n).groupby(uf).sum()[UFS_].to_numpy(); uf_reg = pd.Series(reg).groupby(uf).first()[UFS_].to_numpy()
def resumo(sigma):
    V = simular(sigma)
    linhas = {}
    for u_i, u in enumerate(UFS_): linhas[("uf", u)] = (V[:, u_i], n_uf[u_i], obs_uf[u])
    for r in REGS:
        m = uf_reg == r; linhas[("regiao", r)] = (V[:, m].sum(1), n_uf[m].sum(), obs_reg[r])
    linhas[("regiao", "Brasil")] = (V.sum(1), n_uf.sum(), y.sum() / n.sum())
    out = []
    for (nivel, nome), (v, nn, real) in linhas.items():
        q = np.quantile(v / nn, [0.025, 0.5, 0.975])
        pt = (prev_uf[UFS_].values * n_uf)[uf_reg == nome].sum() / nn if nivel == "regiao" and nome != "Brasil" else \
             (prev_uf[nome] if nivel == "uf" else (prev_uf[UFS_].values * n_uf).sum() / nn)
        out.append(dict(nivel=nivel, nome=nome, lula_pct=pt * 100, q025=q[0] * 100, q50=q[1] * 100, q975=q[2] * 100, real_pct=real * 100,
                        flavio_pct=100 - pt * 100, flavio_q025=100 - q[2] * 100, flavio_q975=100 - q[0] * 100, flavio_real_pct=100 - real * 100))
    d = pd.DataFrame(out); d["erro_pp"] = d.lula_pct - d.real_pct; d["dentro_ic95"] = (d.real_pct >= d.q025) & (d.real_pct <= d.q975)
    d["largura_pp"] = d.q975 - d.q025
    return d
EST = {"RMS": resumo(SIG_RMS), "GLMM": resumo(SIG_GLMM), "zero": resumo(0.0)}
P = EST["RMS"]
P[P.nivel == "regiao"].drop(columns="nivel").rename(columns={"nome": "regiao"}).to_csv(TAB / "estimativas_regionais.csv", index=False)
P[P.nivel == "uf"].drop(columns="nivel").rename(columns={"nome": "uf"}).to_csv(TAB / "estimativas_uf.csv", index=False)
print(P[P.nivel == "regiao"][["nome", "lula_pct", "q025", "q975", "real_pct", "erro_pp", "dentro_ic95", "largura_pp", "flavio_pct", "flavio_real_pct"]].to_string(index=False))

        nome  lula_pct    q025    q975  real_pct  erro_pp  dentro_ic95  largura_pp  flavio_pct  flavio_real_pct
Centro-Oeste   40.3231 38.6132 41.9708   39.9912   0.3319         True      3.3576     59.6769          60.0088
    Nordeste   69.4154 68.5025 70.2660   69.3685   0.0469         True      1.7635     30.5846          30.6315
       Norte   49.4973 48.1017 51.0272   50.0054  -0.5081         True      2.9255     50.5027          49.9946
     Sudeste   47.7116 46.1075 49.4676   47.2419   0.4697         True      3.3601     52.2884          52.7581
         Sul   36.9961 35.5077 38.4605   37.4239  -0.4277         True      2.9528     63.0039          62.5761
      Brasil   51.9070 51.1398 52.6289   51.7796   0.1274         True      1.4892     48.0930          48.2204


In [8]:
cob = []
for k, d in EST.items():
    u, r = d[d.nivel == "uf"], d[(d.nivel == "regiao") & (d.nome != "Brasil")]
    cob.append(dict(sigma=k, valor=dict(RMS=SIG_RMS, GLMM=SIG_GLMM, zero=0.0)[k], cob_uf=u.dentro_ic95.mean(), n_uf_dentro=int(u.dentro_ic95.sum()),
                    cob_regiao=r.dentro_ic95.mean(), n_reg_dentro=int(r.dentro_ic95.sum()), largura_media_uf_pp=u.largura_pp.mean(), largura_media_reg_pp=r.largura_pp.mean()))
print(pd.DataFrame(cob).to_string(index=False)); print("(cobertura nominal 95%; 27 UFs, 5 regiões)")
print("\nUFs fora do IC95 (σ RMS):", list(P[(P.nivel == "uf") & ~P.dentro_ic95].nome), "| erro absoluto médio UF:", round(P[P.nivel == "uf"].erro_pp.abs().mean(), 3), "p.p. | regiões:", round(P[(P.nivel == "regiao") & (P.nome != "Brasil")].erro_pp.abs().mean(), 3), "p.p.")
print("Brasil:", P[P.nome == "Brasil"][["lula_pct", "q025", "q975", "real_pct"]].round(3).to_string(index=False, header=False))
# Boa Esperança do Norte (MT, fora do modelo): os CSVs oficiais já coincidem com as linhas modeladas; efeito de somá-la:
rr = pd.read_csv(TAB / "resultado_regional.csv"); rr = rr[rr.ano == rr.ano.max()].set_index("regiao")
mt = pd.DataFrame({"y": y, "n": n, "r": reg}).groupby("r").sum()
print("max |oficial - modelado| (votos PT; válidos):", int((rr.votos_pt - mt.y).abs().max()), int((rr.n - mt.n).abs().max()))
co = mt.loc["Centro-Oeste"]; co_ben = (co.y + 945) / (co.n + 945 + 2353) * 100
br_ben = (y.sum() + 945) / (n.sum() + 945 + 2353) * 100
print(f"Boa Esperança do Norte (PT 945, Flávio 2.353): Centro-Oeste {co.y/co.n*100:.4f}% -> {co_ben:.4f}% ({co_ben-co.y/co.n*100:+.5f} p.p.); Brasil {y.sum()/n.sum()*100:.4f}% -> {br_ben:.4f}% ({br_ben-y.sum()/n.sum()*100:+.5f} p.p.)")
fig = go.Figure()
d = P[P.nivel == "regiao"].sort_values("lula_pct")
fig.add_trace(go.Scatter(x=d.lula_pct, y=d.nome, mode="markers", name="previsto (LOSO)", error_x=dict(type="data", symmetric=False, array=d.q975 - d.lula_pct, arrayminus=d.lula_pct - d.q025), marker=dict(color="#1b6ca8", size=9)))
fig.add_trace(go.Scatter(x=d.real_pct, y=d.nome, mode="markers", name="real", marker=dict(color="#c2571a", symbol="diamond", size=10)))
fig.update_layout(title="Share de Lula por região: previsto fora da amostra (IC95) e real", xaxis_title="% dos votos válidos", template="plotly_white", height=420)
fig.write_html(FIG / "cv_estimativas_regionais.html", include_plotlyjs="cdn")
u = P[P.nivel == "uf"].sort_values("real_pct")
fig = go.Figure([go.Scatter(x=u.nome, y=u.q975, mode="lines", line=dict(width=0), showlegend=False), go.Scatter(x=u.nome, y=u.q025, mode="lines", line=dict(width=0), fill="tonexty", fillcolor="rgba(27,108,168,.25)", name="IC95"),
                 go.Scatter(x=u.nome, y=u.lula_pct, mode="markers", name="previsto", marker=dict(color="#1b6ca8")), go.Scatter(x=u.nome, y=u.real_pct, mode="markers", name="real", marker=dict(color="#c2571a", symbol="diamond"))])
fig.update_layout(title="Share de Lula por UF: previsto (LOSO) e real", yaxis_title="%", template="plotly_white", height=460)
fig.write_html(FIG / "cv_estimativas_uf.html", include_plotlyjs="cdn")
print("tempo total:", round(time.time() - T0), "s")

sigma  valor  cob_uf  n_uf_dentro  cob_regiao  n_reg_dentro  largura_media_uf_pp  largura_media_reg_pp
  RMS 0.0536  1.0000           27      1.0000             5               5.2861                2.8719
 GLMM 0.0532  1.0000           27      1.0000             5               5.2902                2.8210
 zero 0.0000  0.7407           20      0.8000             4               2.4326                1.1638
(cobertura nominal 95%; 27 UFs, 5 regiões)

UFs fora do IC95 (σ RMS): [] | erro absoluto médio UF: 1.0 p.p. | regiões: 0.357 p.p.
Brasil: 51.9070 51.1400 52.6290 51.7800
max |oficial - modelado| (votos PT; válidos): 0 0
Boa Esperança do Norte (PT 945, Flávio 2.353): Centro-Oeste 39.9912% -> 39.9872% (-0.00400 p.p.); Brasil 51.7796% -> 51.7790% (-0.00062 p.p.)
tempo total: 35 s


_(conclusões preenchidas após a execução)_

## Sensibilidade das estimativas regionais ao modelo

O erro LOSO **regional** não é uma métrica justa para escolher modelo: num binomial de ligação canônica (logit) com dummies de região, a equação de escore de cada dummy
força Σ_u w_u δ_u = 0 (δ_u = resíduo de share da UF u dentro da amostra, w_u = seu peso em votos na região). Ao reter a UF u, o intercepto da região se move em
Δ = −w_u δ_u / (1 − w_u), e o erro retido da UF fica ≈ δ_u / (1 − w_u). O erro regional LOSO é Σ_u w_u δ_u / (1 − w_u): com pesos iguais vale
(1/(1 − w)) Σ_u w δ_u = 0 exatamente; com pesos desiguais fica perto de zero (a aproximação ignora a mudança nos demais coeficientes). O probit satisfaz as equações
ponderadas só aproximadamente. O beta-binomial pesa cada município por n/(1 + (n−1)ρ), então seu escore não é ponderado por votos e o cancelamento não ocorre.
Um erro regional LOSO pequeno num binomial canônico revela esse mecanismo, não um modelo melhor. A métrica de seleção justa é o erro por UF.

A tabela abaixo compara a estimativa pontual LOSO de `bin_probit`, `bin_logit` e `bb_backward` (regra de escolha mantida: `bb_backward`). Também mostra o nº de UFs
e o maior peso w_u por região, que indica onde o cancelamento não é exato (SP no Sudeste).

In [9]:
MODS = ["bin_probit", "bin_logit", "bb_backward"]
obs_r = obs_reg.copy(); obs_r["Brasil"] = y.sum() / n.sum()
w_uf = pd.Series(n).groupby([reg, uf]).sum()
w_uf = w_uf / w_uf.groupby(level=0).sum()
SENS = pd.DataFrame({"real_pct": obs_r * 100})
for m in MODS:
    est = M.agregar(PIL[m], n, reg) * 100
    SENS[f"{m}_pct"] = est; SENS[f"{m}_erro_pp"] = est - SENS["real_pct"]
SENS = SENS.loc[sorted(set(SENS.index) - {"Brasil"}) + ["Brasil"]]
SENS.insert(1, "n_ufs", [int(pd.Series(uf)[reg == r].nunique()) if r != "Brasil" else int(pd.Series(uf).nunique()) for r in SENS.index])
wm = [w_uf[r].max() if r != "Brasil" else (pd.Series(n).groupby(uf).sum() / n.sum()).max() for r in SENS.index]
SENS.insert(2, "w_max_uf", wm)
SENS.index.name = "regiao"
SENS.reset_index().to_csv(TAB / "sensibilidade_regional.csv", index=False)
print(SENS.round(3).to_string())
cv = pd.read_csv(TAB / "cv_loso.csv").set_index("modelo").loc[MODS]
print(cv[["mae_uf_pp", "mae_reg_pp", "max_reg_pp"]].round(3).to_string())
rg = SENS.drop("Brasil")
est_cols = [f"{m}_pct" for m in MODS]
print("amplitude entre modelos (máx - mín) por região, p.p.:", (rg[est_cols].max(axis=1) - rg[est_cols].min(axis=1)).round(3).to_dict())
print("Brasil:", round(float(SENS.loc["Brasil", est_cols].max() - SENS.loc["Brasil", est_cols].min()), 3))

              real_pct  n_ufs  w_max_uf  bin_probit_pct  bin_probit_erro_pp  bin_logit_pct  bin_logit_erro_pp  bb_backward_pct  bb_backward_erro_pp
regiao                                                                                                                                             
Centro-Oeste   39.9910      4    0.4210         39.9950              0.0040        39.9840            -0.0080          40.3230               0.3320
Nordeste       69.3690      9    0.2580         69.4520              0.0840        69.4560             0.0870          69.4150               0.0470
Norte          50.0050      7    0.4870         50.1500              0.1450        50.1730             0.1680          49.4970              -0.5080
Sudeste        47.2420      4    0.5160         47.3850              0.1430        47.3920             0.1500          47.7120               0.4700
Sul            37.4240      3    0.3770         37.4240             -0.0000        37.4260             0.0020   